# Imports

In [1]:
import os
import json
from langchain_text_splitters import RecursiveCharacterTextSplitter

# --- 1. LOADING THE DATA ---
# Loading the clean JSON document.
# Note to myself: JSON is much better than raw .txt because I can easily target just the 'odôvodnenie' (reasoning)
# and ignore the useless legal boilerplate in 'poučenie'.
file_path = "../data/02_processed_json/KS_Banská_Bystrica_41Cob_3_2025_00_dokument.json"
with open(file_path, "r", encoding="utf-8") as f:
    doc_data = json.load(f)

# Extracting only the relevant text and original metadata
reasoning_text = doc_data["segments"]["reasoning"]
base_metadata = doc_data["metadata"]

# --- 2. DEFINING STRATEGIES (The core of my bachelor thesis research) ---

# STRATEGY A: For OpenAI text-embedding-3-small
# I learned that modern API models have a huge context window (8192 tokens).
# So I don't need to artificially break long sentences. I can just split by whole paragraphs (\n\n).
# This keeps the semantic meaning 100% intact.
text_splitter_openai = RecursiveCharacterTextSplitter(
    separators=[r"\n\n"], # Only hard paragraphs! No sentence breaking.
    chunk_size=4000,      # Huge chunk size, safe for OpenAI's token limit
    chunk_overlap=200,    # A bit of overlap just in case the context flows to the next paragraph
    length_function=len,
    is_separator_regex=True
)

# STRATEGY B: For local open-source mE5-small (Microsoft intfloat)
# Local models have a strict limit of 512 tokens. 
# 1200 characters is my safe zone for Slovak language (approx. 3-4 chars per token).
# I must use smart regex here, otherwise it will brutally cut numbers like "1." or "0,5 %".
text_splitter_me5 = RecursiveCharacterTextSplitter(
    separators=[
        r"\n\n",                                  # 1. Paragraph (odsek)
        r"\n",                                    # 2. New line (nový riadok)
        r"(?<=[a-zA-Zá-žÁ-Ž])\.\s+(?=[A-ZÁ-Ž])",  # 3. Smart dot - cuts only at the real end of a sentence
        r";\s+",                                  # 4. Semicolon - good for long legal lists
        r",\s+",                                  # 5. Smart comma - saves decimals (0,5) but breaks long sentences
        r"\s+"                                    # 6. Fallback space
    ],
    chunk_size=1200,      # Strict limit!
    chunk_overlap=150,
    length_function=len,
    is_separator_regex=True # CRITICAL: Must be True for regex to work!
)

# --- 3. PROCESSING THE TEXT ---
# Helper function to process and pack chunks with metadata
def process_chunks(splitter, text, metadata, strategy_name):
    raw_chunks = splitter.split_text(text)
    final_chunks = []
    
    for i, chunk_txt in enumerate(raw_chunks):
        chunk_meta = metadata.copy()
        chunk_meta["chunk_index"] = i
        chunk_meta["source_file"] = doc_data["filename"]
        chunk_meta["strategy"] = strategy_name # Important for my evaluation later
        
        final_chunks.append({
            "page_content": chunk_txt,
            "metadata": chunk_meta
        })
    return final_chunks

# Generating both datasets
chunks_openai = process_chunks(text_splitter_openai, reasoning_text, base_metadata, "openai_api_4000")
chunks_me5 = process_chunks(text_splitter_me5, reasoning_text, base_metadata, "me5_local_1200")

# --- 4. SAVING THE DATASETS ---
OUTPUT_DIR = "../data/03_chunked_docs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save OpenAI version
path_openai = os.path.join(OUTPUT_DIR, doc_data["filename"].replace(".pdf", "_chunks_OPENAI.json"))
with open(path_openai, "w", encoding="utf-8") as f:
    json.dump(chunks_openai, f, ensure_ascii=False, indent=4)

# Save mE5 version
path_me5 = os.path.join(OUTPUT_DIR, doc_data["filename"].replace(".pdf", "_chunks_ME5.json"))
with open(path_me5, "w", encoding="utf-8") as f:
    json.dump(chunks_me5, f, ensure_ascii=False, indent=4)

# --- 5. PRINTING THE RESULTS FOR MY THESIS LOGS ---
print("=== CHUNKING PROCESS COMPLETED ===")
print(f"Original reasoning length: {len(reasoning_text)} characters.")
print(f"-> STRATEGY A (OpenAI): Created {len(chunks_openai)} chunks. Saved to {path_openai}")
print(f"-> STRATEGY B (mE5): Created {len(chunks_me5)} chunks. Saved to {path_me5}")

# Quick sanity check
print("\n[Sanity Check] First chunk of OpenAI strategy:")
print(f"Length: {len(chunks_openai[0]['page_content'])} chars. Starts with: {chunks_openai[0]['page_content'][:50]}...")

=== CHUNKING PROCESS COMPLETED ===
Original reasoning length: 33330 characters.
-> STRATEGY A (OpenAI): Created 11 chunks. Saved to ../data/03_chunked_docs/KS_Banská_Bystrica_41Cob_3_2025_00_dokument_chunks_OPENAI.json
-> STRATEGY B (mE5): Created 43 chunks. Saved to ../data/03_chunked_docs/KS_Banská_Bystrica_41Cob_3_2025_00_dokument_chunks_ME5.json

[Sanity Check] First chunk of OpenAI strategy:
Length: 3533 chars. Starts with: odôvodnenie :

1. Súd prvej inštancie napadnutým r...
